# Truy xuat thong tin ket hop KNN

## 1. Xay dung truy van su dung Mo hinh xac suat

In [74]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [75]:
def preprocess(tok, stemmer=ps, punctlist=puncts, stopwords=stoplist):
  tok = tok.lower()
  if tok.isdigit():
    return None
  if tok.isnumeric():
    return None
  if tok in punctlist:
    return None
  if tok in stopwords:
    return None
  return stemmer.stem(tok)

def indexing(src, idx="ind"):
  if src[-1] != '/':
    src += '/'
  schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=StandardAnalyzer()))
  os.makedirs(idx, exist_ok=True)
  ix = create_in(idx, schema)
  writer = ix.writer()

  files = os.listdir(src)
  for f in files:
    r = open(src + f, encoding="cp1252")
    terms = []
    for s in r:
      for sent in sent_tokenize(s.strip()):
        for tok in word_tokenize(sent):
          tok = preprocess(tok)
          if tok != None:
            terms.append(tok)
    r.close()
    cont = " ".join(terms)
    writer.add_document(docid="{}".format(f.split(".")[0]), content=cont)
  writer.commit()

In [76]:
import os
import shutil
import stat

def force_delete_directory(path):
    def onerror(func, path, exc_info):
        os.chmod(path, stat.S_IWRITE)
        func(path)

    if os.path.exists(path):
        shutil.rmtree(path, onerror=onerror)
        print(f"Đã xóa thư mục: {path}")

force_delete_directory('ind')

Đã xóa thư mục: ind


In [77]:
indexing("../Cranfield/Cranfield", "ind")

In [78]:
def readGroundTruth(src):
  if src[-1] != '/':
    src += '/'

  GT = {}
  for f in os.listdir(src):
    r = open(src + f)
    rel = {}
    for s in r:
      s = s.strip()
      sp = s.split("\t")
      if len(sp) < 2:
        continue
      did = sp[0].split(" ")[1]
      rel[did] = int(sp[1])
    GT[f.split(".")[0]] = rel
    r.close()
  return GT

In [79]:
GroundTruth = readGroundTruth("../Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [80]:
def readQuery(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        tok = preprocess(tok)
        if tok != None:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [81]:
Queries = readQuery("../Cranfield/query.txt")
print(Queries)

{'1': 'similar law must obey construct aeroelast model heat high speed aircraft', '2': 'structur aeroelast problem associ flight high speed aircraft', '3': 'problem heat conduct composit slab solv far', '4': 'criterion develop show empir valid flow solut chemic react ga mixtur base simplifi assumpt instantan local chemic equilibrium', '5': 'chemic kinet system applic hyperson aerodynam problem', '6': 'theoret experiment guid turbul couett flow behaviour', '7': 'possibl relat avail pressur distribut ogiv forebodi zero angl attack lower surfac pressur equival ogiv forebodi angl attack', '8': 'method -dash exact approxim -dash present avail predict bodi pressur angl attack', '9': 'paper intern /slip flow/ heat transfer studi', '10': 'real-ga transport properti air avail wide rang enthalpi densiti', '11': 'possibl find analyt similar solut strong blast wave problem newtonian approxim', '12': 'aerodynam perform channel flow ground effect machin calcul', '13': 'basic mechan transon aileron b

In [82]:
def processQueries(ind, qry):

  idx = index.open_dir(ind)
  searcher = idx.searcher(weighting=scoring.BM25F())
  parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

  RET = {}
  for key in qry:
    query = parser.parse(qry[key])
    results = searcher.search(query, limit=None)
    rel = {}
    for i in range(len(results)):
      rel[results[i]["docid"]] = results[i].score
    RET[key] = rel
  return RET

In [83]:
RunResults = processQueries("ind", Queries)

In [84]:
RunResults

{'1': {'51': 30.582149193172796,
  '486': 28.83579111379974,
  '12': 25.50537784336174,
  '878': 23.32185350009788,
  '184': 22.561495282152045,
  '573': 21.46562235452518,
  '665': 18.798451539637913,
  '141': 17.661696726070065,
  '746': 17.176007182081598,
  '747': 16.47038542244308,
  '944': 16.384715541674765,
  '435': 16.34356872033145,
  '876': 16.158111826985586,
  '13': 16.05474203271634,
  '78': 15.955933627999888,
  '879': 15.935878648199285,
  '1263': 15.180230311795489,
  '453': 15.046807862639223,
  '14': 15.040218243688633,
  '663': 14.905931133647336,
  '172': 14.801955386048787,
  '1268': 14.788368777332456,
  '329': 14.749458137198744,
  '252': 14.745869033424214,
  '1361': 14.670226436827168,
  '1003': 14.26661995836298,
  '219': 13.799021235902046,
  '359': 13.486717037126775,
  '332': 13.21194660116188,
  '1328': 13.117947826647972,
  '792': 13.0600542041816,
  '195': 13.052395051049075,
  '1144': 13.050635664992589,
  '576': 13.039666668154677,
  '293': 12.7762283

In [85]:
print(pytrec_eval.supported_measures)
eval = pytrec_eval.RelevanceEvaluator(GroundTruth, ["infAP", "11pt_avg"])
Results = eval.evaluate(RunResults)

MAP = 0
MAP11PT = 0

for key in Results:
  value = Results[key]["infAP"]
  if not math.isnan(value):
    MAP += value
  value = Results[key]["11pt_avg"]
  if not math.isnan(value):
    MAP11PT += value

MAP /= len(Results)
MAP11PT /= len(Results)

print(MAP, MAP11PT)

{'binG', 'set_F', 'recip_rank', 'set_P', 'ndcg', 'ndcg_rel', 'num_nonrel_judged_ret', 'infAP', 'gm_map', 'set_map', 'Rprec_mult', 'relstring', 'success', 'num_q', 'num_rel_ret', 'Rprec', 'map', 'relative_P', 'set_recall', 'map_cut', 'num_rel', 'runid', 'gm_bpref', 'G', 'iprec_at_recall', 'bpref', 'utility', 'Rndcg', 'P', 'recall', 'num_ret', 'set_relative_P', '11pt_avg', 'ndcg_cut'}
0.35013425744507476 0.3213571224969277


## 2. Mo hinh KNN

### 1. Chuan bi du lieu KNN (TF-IDF)

In [86]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Lay toan bo tai lieu da tien xu ly

In [87]:
def load_docs_for_knn(src):
    docs = {}
    for f in os.listdir(src):
        r = open(os.path.join(src, f), encoding="cp1252")
        terms = []
        for s in r:
            for sent in sent_tokenize(s.strip()):
                for tok in word_tokenize(sent):
                    tok = preprocess(tok)
                    if tok:
                        terms.append(tok)
        r.close()
        docs[f.split(".")[0]] = " ".join(terms)
    return docs

Docs = load_docs_for_knn("../Cranfield/Cranfield")


### 2. Xay dung mo hinh KNN (TF-IDF + Cosine)

In [88]:
doc_ids = list(Docs.keys())
doc_texts = list(Docs.values())

vectorizer = TfidfVectorizer()
X_docs = vectorizer.fit_transform(doc_texts)


### 3. Truy van bang KNN

In [121]:
def processQueries_KNN(queries, topk=1000):
    RET = {}

    for qid, qtext in queries.items():
        q_vec = vectorizer.transform([qtext])
        sims = cosine_similarity(q_vec, X_docs)[0]

        ranked = sorted(
            zip(doc_ids, sims),
            key=lambda x: x[1],
            reverse=True
        )

        rel = {}
        for docid, score in ranked[:topk]:
            if score > 0:
                rel[docid] = float(score)

        RET[qid] = rel
    return RET

RunResults_KNN = processQueries_KNN(Queries)

In [122]:
print(pytrec_eval.supported_measures)
eval = pytrec_eval.RelevanceEvaluator(GroundTruth, ["infAP", "11pt_avg"])
Results = eval.evaluate(RunResults_KNN)

MAP = 0
MAP11PT = 0

for key in Results:
  value = Results[key]["infAP"]
  if not math.isnan(value):
    MAP += value
  value = Results[key]["11pt_avg"]
  if not math.isnan(value):
    MAP11PT += value

MAP /= len(Results)
MAP11PT /= len(Results)

print(MAP, MAP11PT)

{'binG', 'set_F', 'recip_rank', 'set_P', 'ndcg', 'ndcg_rel', 'num_nonrel_judged_ret', 'infAP', 'gm_map', 'set_map', 'Rprec_mult', 'relstring', 'success', 'num_q', 'num_rel_ret', 'Rprec', 'map', 'relative_P', 'set_recall', 'map_cut', 'num_rel', 'runid', 'gm_bpref', 'G', 'iprec_at_recall', 'bpref', 'utility', 'Rndcg', 'P', 'recall', 'num_ret', 'set_relative_P', '11pt_avg', 'ndcg_cut'}
0.33418408425431056 0.3091996576968329


## Ket hop 2 mo hinh de truy van

Chuẩn hoá score (min-max)

In [123]:
def normalize_scores(run):
    norm_run = {}
    for qid, docs in run.items():
        if not docs:
            norm_run[qid] = {}
            continue
        scores = list(docs.values())
        min_s, max_s = min(scores), max(scores)
        norm_run[qid] = {}
        for d, s in docs.items():
            if max_s > min_s:
                norm_run[qid][d] = (s - min_s) / (max_s - min_s)
            else:
                norm_run[qid][d] = 0.0
    return norm_run


Fusion

In [124]:
def fuse_runs(bm25, knn, alpha=0.6):
    bm25 = normalize_scores(bm25)
    knn = normalize_scores(knn)

    FUSED = {}

    for qid in bm25:
        docs = set(bm25[qid].keys()) | set(knn.get(qid, {}).keys())
        fused_scores = {}
        for d in docs:
            fused_scores[d] = (
                alpha * bm25[qid].get(d, 0) +
                (1 - alpha) * knn.get(qid, {}).get(d, 0)
            )
        FUSED[qid] = fused_scores

    return FUSED


In [125]:
RunResults_Fused = fuse_runs(RunResults, RunResults_KNN, alpha=0.6)

In [ ]:
import pytrec_eval

def print_best_worst_queries(
    RunResults,
    Queries,
    GroundTruth,
    top_k=5,
    retrieved_k=10
):
    evaluator = pytrec_eval.RelevanceEvaluator(GroundTruth, {"map"})
    results = evaluator.evaluate(RunResults)

    # Lọc query hợp lệ (MAP != NaN)
    valid = [
        (qid, res["map"])
        for qid, res in results.items()
        if not math.isnan(res["map"])
    ]

    # Sort theo MAP
    valid_sorted = sorted(valid, key=lambda x: x[1], reverse=True)

    best = valid_sorted[:top_k]
    worst = valid_sorted[-top_k:]

    def print_block(title, items):
        print("\n" + "=" * 60)
        print(title)
        print("=" * 60)

        for qid, map_score in items:
            print(f"\nQuery ID : {qid}")
            print(f"Query    : {Queries[qid]}")
            print(f"MAP      : {map_score:.4f}")

            # Relevant docs
            rel_docs = [
                docid for docid, rel in GroundTruth[qid].items()
                if rel > 0
            ]
            print(f"Relevant docs ({len(rel_docs)}): {rel_docs[:retrieved_k]}")

            # Retrieved docs
            retrieved = list(RunResults[qid].keys())[:retrieved_k]
            print(f"Top retrieved docs: {retrieved}")

    print_block("🔥 TOP QUERIES (Highest MAP)", best)
    print_block("❄️ WORST QUERIES (Lowest MAP)", worst)


In [146]:
print_best_worst_queries(
    RunResults_Fused,
    Queries,
    GroundTruth,
    top_k=5,
    retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 15
Query    : materi properti photoelast materi
MAP      : 1.0000
Relevant docs (2): ['463', '462']
Top retrieved docs: ['927', '1263', '658', '509', '364', '42', '974', '740', '1287', '81']

Query ID : 119
Query    : effect initi axisymmetr deviat circular non linear ( large-deflect ) load-deflect respons cylind hydrostat pressur
MAP      : 1.0000
Relevant docs (1): ['926']
Top retrieved docs: ['1275', '898', '737', '57', '692', '846', '714', '920', '42', '974']

Query ID : 185
Query    : experiment studi panel flutter
MAP      : 0.8488
Relevant docs (9): ['858', '859', '857', '1008', '856', '15', '285', '894', '766']
Top retrieved docs: ['927', '919', '1263', '497', '464', '846', '967', '42', '721', '974']

Query ID : 41
Query    : anyon investig develop simpl model vortex wake behind cruciform wing
MAP      : 0.8333
Relevant docs (3): ['288', '289', '433']
Top retrieved docs: ['57', '692', '714', '920', '42', '974', '996', '266', '243', '50']

## Danh gia

### Truy van ban dau

In [126]:
print(pytrec_eval.supported_measures)
eval = pytrec_eval.RelevanceEvaluator(GroundTruth, ["infAP", "11pt_avg"])
Results = eval.evaluate(RunResults)

MAP = 0
MAP11PT = 0

for key in Results:
  value = Results[key]["infAP"]
  if not math.isnan(value):
    MAP += value
  value = Results[key]["11pt_avg"]
  if not math.isnan(value):
    MAP11PT += value

MAP /= len(Results)
MAP11PT /= len(Results)

print(MAP, MAP11PT)

{'binG', 'set_F', 'recip_rank', 'set_P', 'ndcg', 'ndcg_rel', 'num_nonrel_judged_ret', 'infAP', 'gm_map', 'set_map', 'Rprec_mult', 'relstring', 'success', 'num_q', 'num_rel_ret', 'Rprec', 'map', 'relative_P', 'set_recall', 'map_cut', 'num_rel', 'runid', 'gm_bpref', 'G', 'iprec_at_recall', 'bpref', 'utility', 'Rndcg', 'P', 'recall', 'num_ret', 'set_relative_P', '11pt_avg', 'ndcg_cut'}
0.35013425744507476 0.3213571224969277


In [127]:
def average_precision_at_k(ranked_docs, relevant_docs, k=20):
    score = 0.0
    hits = 0
    for i, doc_id in enumerate(ranked_docs[:k], start=1):
        if doc_id in relevant_docs:
            hits += 1
            score += hits / i
    if not relevant_docs:
        return 0.0
    return score / min(len(relevant_docs), k)


MAP20 = 0
for qid in RunResults:
    ranked_docs = list(RunResults[qid].keys())
    relevant_docs = {
        doc_id for doc_id, rel in GroundTruth[qid].items() if rel > 0
    }
    MAP20 += average_precision_at_k(ranked_docs, relevant_docs, k=10)

MAP20 /= len(RunResults)
print("MAP@10 =", MAP20)

MAP@10 = 0.2509913202317964


### Ket hop KNN

In [96]:
Results_Fused = eval.evaluate(RunResults_Fused)

In [128]:
import math
import pytrec_eval

metrics = [
    "map",
    "infAP",
    "ndcg",
    "11pt_avg",
    "P_5", "P_10", "P_20",
    "recall_5", "recall_10", "recall_20"
]

evaluator = pytrec_eval.RelevanceEvaluator(GroundTruth, metrics)
results = evaluator.evaluate(RunResults_Fused)

# Khởi tạo
MAP = infAP = MAP11PT = 0.0
P5 = P10 = P20 = 0.0
R5 = R10 = R20 = 0.0

valid_queries = 0

for qid, res in results.items():
    if not math.isnan(res["map"]):
        valid_queries += 1

        MAP += res["map"]
        infAP += res["infAP"]
        MAP11PT += res["11pt_avg"]

        P5 += res["P_5"]
        P10 += res["P_10"]
        P20 += res["P_20"]

        R5 += res["recall_5"]
        R10 += res["recall_10"]
        R20 += res["recall_20"]

# Trung bình
MAP /= valid_queries
infAP /= valid_queries
MAP11PT /= valid_queries

P5 /= valid_queries
P10 /= valid_queries
P20 /= valid_queries

R5 /= valid_queries
R10 /= valid_queries
R20 /= valid_queries

# 🔹 TÍNH F1@k
def f1(p, r):
    return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)

F1_5 = f1(P5, R5)
F1_10 = f1(P10, R10)
F1_20 = f1(P20, R20)


In [129]:
print("=== BM25 + KNN Fusion ===")
print(f"Queries evaluated : {valid_queries}")

print(f"MAP        : {MAP:.4f}")
print(f"infAP      : {infAP:.4f}")
print(f"11pt Avg   : {MAP11PT:.4f}")

print(f"P@5        : {P5:.4f}")
print(f"R@5        : {R5:.4f}")
print(f"F1@5       : {F1_5:.4f}")

print(f"P@10       : {P10:.4f}")
print(f"R@10       : {R10:.4f}")
print(f"F1@10      : {F1_10:.4f}")

print(f"P@20       : {P20:.4f}")
print(f"R@20       : {R20:.4f}")
print(f"F1@20      : {F1_20:.4f}")


=== BM25 + KNN Fusion ===
Queries evaluated : 225
MAP        : 0.3100
infAP      : 0.3644
11pt Avg   : 0.3341
P@5        : 0.3156
R@5        : 0.2849
F1@5       : 0.2995
P@10       : 0.2369
R@10       : 0.3961
F1@10      : 0.2965
P@20       : 0.1636
R@20       : 0.5204
F1@20      : 0.2489


In [130]:
# Tính trung bình (macro average) cho tất cả query
avg_metrics = {}
for m in next(iter(results.values())).keys():
    avg_metrics[m] = sum(q[m] for q in results.values()) / len(results)

# Tính macro F1 từ Precision và Recall trung bình
cutoffs = [5, 10, 20]

# Tính macro F1 cho các cutoff
for k in cutoffs:
    p_key = f"P_{k}"
    r_key = f"recall_{k}"
    f1_key = f"F1_{k}"
    
    if p_key in avg_metrics and r_key in avg_metrics:
        p = avg_metrics[p_key]
        r = avg_metrics[r_key]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        avg_metrics[f1_key] = f1

# In ra
print("Average over all queries:")
for m, v in avg_metrics.items():
    print(f"  {m:10s}: {v:.4f}")

Average over all queries:
  map       : 0.3100
  P_5       : 0.3156
  P_10      : 0.2369
  P_20      : 0.1636
  recall_5  : 0.2849
  recall_10 : 0.3961
  recall_20 : 0.5204
  infAP     : 0.3644
  11pt_avg  : 0.3341
  ndcg      : 0.5118
  F1_5      : 0.2995
  F1_10     : 0.2965
  F1_20     : 0.2489


### KNN lan 2


### 1. Vector hóa Document (TF-IDF – đơn giản nhất)

In [100]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

In [101]:
def load_docs(src):
    docs = {}
    for f in os.listdir(src):
        with open(src + '/' + f, encoding="cp1252") as r:
            terms = []
            for s in r:
                for sent in sent_tokenize(s.strip()):
                    for tok in word_tokenize(sent):
                        tok = preprocess(tok)
                        if tok:
                            terms.append(tok)
            docs[f.split(".")[0]] = " ".join(terms)
    return docs

docs = load_docs("../Cranfield/Cranfield")


In [102]:
vectorizer = TfidfVectorizer()
doc_ids = list(docs.keys())
X = vectorizer.fit_transform(docs.values())

# Lưu lại
pickle.dump((vectorizer, X, doc_ids), open("tfidf.pkl", "wb"))


### 2. Xây dựng KNN

In [103]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=50,
    metric="cosine"
)
knn.fit(X)

,n_neighbors,50
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


### 3. Kết hợp BM25 + KNN khi truy vấn

In [132]:
def query_vectorize(query, vectorizer):
    return vectorizer.transform([query])

In [131]:
def hybrid_search_doc_knn(
    bm25_results,        # dict {docid: bm25_score} đã sort
    X,                   # document vectors
    doc_ids,             # index -> docid
    knn,                 # fitted NearestNeighbors on X
    top_bm25=50,
    k_knn=10,
    alpha=0.7
):
    scores = {}

    # 1. Top BM25
    bm25_docs = list(bm25_results.items())[:top_bm25]

    for docid, bm25_score in bm25_docs:
        scores[docid] = alpha * bm25_score

    # map docid -> index
    docid_to_idx = {d: i for i, d in enumerate(doc_ids)}

    # 2. Doc–doc KNN expansion
    for docid, bm25_score in bm25_docs:
        if docid not in docid_to_idx:
            continue

        doc_idx = docid_to_idx[docid]
        doc_vec = X[doc_idx].reshape(1, -1)

        dist, idxs = knn.kneighbors(doc_vec, n_neighbors=k_knn)

        for d, i in zip(dist[0], idxs[0]):
            neighbor_docid = doc_ids[i]
            sim = 1 - d  # cosine similarity

            scores[neighbor_docid] = scores.get(neighbor_docid, 0) + \
                                     (1 - alpha) * sim

    # 3. Sort
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

In [133]:
def compute_MAP(GroundTruth, RunResults):
    evaluator = pytrec_eval.RelevanceEvaluator(
        GroundTruth, {"map"}
    )
    results = evaluator.evaluate(RunResults)

    MAP = 0.0
    valid_queries = 0

    for qid, res in results.items():
        if not math.isnan(res["map"]):
            MAP += res["map"]
            valid_queries += 1

    return MAP / valid_queries if valid_queries > 0 else 0.0


In [134]:
import optuna

In [137]:
def objective(trial):
    # 🔧 Siêu tham số cần tối ưu
    top_bm25 = trial.suggest_int("top_bm25", 20, 100)
    k_knn = trial.suggest_int("k_knn", 5, 50)
    alpha = trial.suggest_float("alpha", 0.0, 1.0)

    RunResults_Fused = {}

    for qid, qtext in Queries.items():
        bm25_res = RunResults[qid]

        fused = hybrid_search_doc_knn(
            bm25_res,
            X,
            doc_ids,
            knn,
            top_bm25=top_bm25,
            k_knn=k_knn,
            alpha=alpha
        )

        # ⚠️ pytrec_eval yêu cầu score là float
        RunResults_Fused[qid] = {
            docid: float(score)
            for docid, score in fused.items()
        }

    # 🎯 Objective = MAP
    return compute_MAP(GroundTruth, RunResults_Fused)

In [138]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(study.best_params)
print(study.best_value)


[I 2025-12-29 03:30:49,248] A new study created in memory with name: no-name-b041b0db-851e-4665-a3eb-1b96ab7618b0
[I 2025-12-29 03:31:22,056] Trial 0 finished with value: 0.29942925130108883 and parameters: {'top_bm25': 49, 'k_knn': 22, 'alpha': 0.8551575295241288}. Best is trial 0 with value: 0.29942925130108883.
[I 2025-12-29 03:31:46,638] Trial 1 finished with value: 0.3118258252905067 and parameters: {'top_bm25': 30, 'k_knn': 35, 'alpha': 0.5234317599972905}. Best is trial 1 with value: 0.3118258252905067.
[I 2025-12-29 03:32:07,699] Trial 2 finished with value: 0.30381790701014333 and parameters: {'top_bm25': 29, 'k_knn': 12, 'alpha': 0.6252504805463796}. Best is trial 1 with value: 0.3118258252905067.
[I 2025-12-29 03:32:47,693] Trial 3 finished with value: 0.29992138876702895 and parameters: {'top_bm25': 63, 'k_knn': 32, 'alpha': 0.3868150444326297}. Best is trial 1 with value: 0.3118258252905067.
[I 2025-12-29 03:33:15,239] Trial 4 finished with value: 0.30977366621747465 and p

{'top_bm25': 26, 'k_knn': 38, 'alpha': 0.40640058848615457}
0.3171034815542735


In [139]:
best_params = study.best_params
best_map = study.best_value

print("Best MAP:", best_map)
print("Best params:", best_params)


Best MAP: 0.3171034815542735
Best params: {'top_bm25': 26, 'k_knn': 38, 'alpha': 0.40640058848615457}


In [ ]:
RunResults_Best = {}

for qid, qtext in Queries.items():
    fused = hybrid_search_doc_knn(
        RunResults[qid],
        qtext,
        vectorizer,
        X,
        doc_ids,
        knn,
        top_bm25=best_params["top_bm25"],
        k_knn=best_params["k_knn"],
        alpha=best_params["alpha"]
    )

    RunResults_Best[qid] = {
        docid: float(score)
        for docid, score in fused.items()
    }


In [112]:
RunResults_Best

{'1': {'51': 1.3922829663706349,
  '486': 1.2097765141119579,
  '12': 1.152573263049365,
  '184': 1.0501796677569266,
  '878': 1.001018450374405,
  '573': 0.9002546125398908,
  '665': 0.8317902566484443,
  '746': 0.7956381062911182,
  '141': 0.765979751826239,
  '435': 0.7424704709910531,
  '879': 0.7424070425234521,
  '944': 0.7391555744235205,
  '13': 0.7366471697000058,
  '876': 0.6950244718565501,
  '747': 0.693746668329706,
  '359': 0.6848032722013438,
  '14': 0.6560667387615107,
  '663': 0.6451077465212589,
  '1361': 0.6337468804958102,
  '56': 0.5991223782332691,
  '792': 0.5824737535098029,
  '332': 0.5787798443338006,
  '1328': 0.5726593949277118,
  '78': 0.5701728101317954,
  '1144': 0.5687943844565331,
  '1186': 0.5459612033696544,
  '1263': 0.5424536587527322,
  '378': 0.538271378915785,
  '453': 0.5376859118728756,
  '875': 0.5312849195796765,
  '172': 0.5289363001046047,
  '1268': 0.5284507932673014,
  '329': 0.5270603519715232,
  '252': 0.5269320981549376,
  '874': 0.515

In [142]:
import math
import pytrec_eval

metrics = [
    "map",
    "infAP",
    "ndcg",
    "11pt_avg",
    "P_5", "P_10", "P_20",
    "recall_5", "recall_10", "recall_20"
]

evaluator = pytrec_eval.RelevanceEvaluator(GroundTruth, metrics)
results = evaluator.evaluate(RunResults_Best)

In [143]:
# Tính trung bình (macro average) cho tất cả query
avg_metrics = {}
for m in next(iter(results.values())).keys():
    avg_metrics[m] = sum(q[m] for q in results.values()) / len(results)

# Tính macro F1 từ Precision và Recall trung bình
cutoffs = [5, 10, 20]

# Tính macro F1 cho các cutoff
for k in cutoffs:
    p_key = f"P_{k}"
    r_key = f"recall_{k}"
    f1_key = f"F1_{k}"
    
    if p_key in avg_metrics and r_key in avg_metrics:
        p = avg_metrics[p_key]
        r = avg_metrics[r_key]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        avg_metrics[f1_key] = f1

# In ra
print("Average over all queries:")
for m, v in avg_metrics.items():
    print(f"  {m:10s}: {v:.4f}")

Average over all queries:
  map       : 0.3005
  P_5       : 0.3138
  P_10      : 0.2373
  P_20      : 0.1616
  recall_5  : 0.2886
  recall_10 : 0.3975
  recall_20 : 0.5167
  infAP     : 0.3542
  11pt_avg  : 0.3247
  ndcg      : 0.4554
  F1_5      : 0.3007
  F1_10     : 0.2972
  F1_20     : 0.2461


In [145]:
print_best_worst_queries(
    RunResults_Best,
    Queries,
    GroundTruth,
    top_k=5,
    retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 15
Query    : materi properti photoelast materi
MAP      : 1.0000
Relevant docs (2): ['463', '462']
Top retrieved docs: ['462', '463', '1025', '1099', '1043', '542', '1340', '1097', '761', '1065']

Query ID : 119
Query    : effect initi axisymmetr deviat circular non linear ( large-deflect ) load-deflect respons cylind hydrostat pressur
MAP      : 1.0000
Relevant docs (1): ['926']
Top retrieved docs: ['926', '897', '744', '1055', '928', '533', '1116', '765', '1033', '952']

Query ID : 185
Query    : experiment studi panel flutter
MAP      : 0.8525
Relevant docs (9): ['858', '859', '857', '1008', '856', '15', '285', '894', '766']
Top retrieved docs: ['856', '857', '766', '1008', '858', '859', '391', '948', '658', '390']

Query ID : 41
Query    : anyon investig develop simpl model vortex wake behind cruciform wing
MAP      : 0.8333
Relevant docs (3): ['288', '289', '433']
Top retrieved docs: ['289', '433', '927', '229', '432', '288', '126', '1152'